# 🧬 ESMC Analysis Pipeline - Colab Version

Interactive protein sequence analysis using ESM-C embeddings.

**Steps:**
1. FASTA Cleaning
2. Embedding Generation
3. Entropy Analysis
4. Logits Analysis
5. Export Results

---

In [ ]:
# ============================================================
# SETUP - Run this first!
# ============================================================

print("🔧 Setting up environment...\n")

import os
import subprocess

# Clone repository
REPO_URL = "https://github.com/espickle1/esmc-analysis-pipeline.git"
REPO_DIR = "esmc-analysis-pipeline"

if not os.path.exists(REPO_DIR):
    print("📥 Cloning repository...")
    result = subprocess.run(["git", "clone", REPO_URL], capture_output=True, text=True)
    if result.returncode != 0:
        print("⚠️ Clone failed - make sure repo is public")
        print(f"Error: {result.stderr}")
        raise Exception("Clone failed")

os.chdir(REPO_DIR)
print(f"📁 Working directory: {os.getcwd()}")

# Install dependencies
print("📦 Installing dependencies...")
!pip install -q esm huggingface_hub ipywidgets pandas torch scikit-learn matplotlib tqdm

# Imports
import sys
from pathlib import Path
import pandas as pd
import torch

# Add src to path
sys.path.insert(0, str(Path.cwd()))

# Verify packages
try:
    from src.embedding import fasta_cleaner
    from src.analysis import entropy_lib
    print("✅ Pipeline packages loaded")
except ImportError as e:
    print(f"❌ Import failed: {e}")

# Check GPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU - running on CPU")

print("\n🎉 Setup complete!")

---
## Step 1: FASTA Cleaning

In [ ]:
from src.embedding.fasta_cleaner import process_fasta_content
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

sequences_df = None
metadata_df = None

fasta_upload = widgets.FileUpload(
    accept=".fasta,.fa,.faa,.txt",
    multiple=True,
    description="Upload FASTA"
)

output = widgets.Output()

def on_upload(change):
    global sequences_df, metadata_df
    with output:
        clear_output()
        if not change["new"]:
            return
        
        print("🔄 Processing...")
        all_seqs, all_meta = [], []
        
        uploaded = change["new"]
        if isinstance(uploaded, dict):
            for fname, finfo in uploaded.items():
                content = finfo["content"].decode("utf-8")
                s, m = process_fasta_content(content, fname)
                all_seqs.append(s)
                all_meta.append(m)
        elif isinstance(uploaded, tuple):
            for finfo in uploaded:
                content = finfo.content.decode("utf-8")
                s, m = process_fasta_content(content, finfo.name)
                all_seqs.append(s)
                all_meta.append(m)
        
        sequences_df = pd.concat(all_seqs, ignore_index=True)
        metadata_df = pd.concat(all_meta, ignore_index=True)
        
        print(f"✅ Processed {len(sequences_df)} sequences")
        display(sequences_df.head())

fasta_upload.observe(on_upload, names="value")

display(HTML("<h3>📁 Upload FASTA Files</h3>"))
display(fasta_upload)
display(output)

In [ ]:
# Save cleaned data
if sequences_df is not None:
    sequences_df.to_csv("data/sequences.csv", index=False)
    metadata_df.to_csv("data/metadata.csv", index=False)
    print("✅ Saved to data/")

---
## Step 2: Embedding Generation

In [ ]:
from src.embedding.esmc_embed_lib import load_esmc_model, embed_sequences, save_embeddings

model = None
embedding_results = None

token_input = widgets.Password(placeholder="HuggingFace token", description="Token:")
model_dropdown = widgets.Dropdown(
    options=[("ESMC 600M", "esmc_600m"), ("ESMC 300M", "esmc_300m")],
    value="esmc_600m",
    description="Model:"
)
load_btn = widgets.Button(description="🔐 Load Model")
embed_btn = widgets.Button(description="🚀 Generate Embeddings")
progress = widgets.IntProgress(min=0, max=100, description="Progress:")
embed_output = widgets.Output()

def on_load(btn):
    global model
    with embed_output:
        clear_output()
        print("🔄 Loading model...")
        model = load_esmc_model(token_input.value, model_dropdown.value)
        print(f"✅ Model loaded on {DEVICE}")

def on_embed(btn):
    global embedding_results
    with embed_output:
        clear_output()
        if model is None:
            print("⚠️ Load model first")
            return
        if sequences_df is None:
            print("⚠️ Upload FASTA first")
            return
        
        print("🔄 Generating embeddings...")
        progress.max = len(sequences_df)
        
        def update(cur, tot):
            progress.value = cur
        
        embedding_results = embed_sequences(
            model, sequences_df,
            return_embeddings=True,
            return_logits=True,
            progress_callback=update
        )
        print(f"✅ Embedded {len(embedding_results['sequence_id'])} sequences")

load_btn.on_click(on_load)
embed_btn.on_click(on_embed)

display(widgets.HBox([token_input, model_dropdown]))
display(widgets.HBox([load_btn, embed_btn]))
display(progress)
display(embed_output)

In [ ]:
# Save embeddings
if embedding_results:
    save_embeddings(embedding_results, "data/embeddings.pt")
    print("✅ Saved data/embeddings.pt")

---
## Step 3: Entropy Analysis

In [ ]:
from src.analysis.entropy_lib import analyze_entropy, entropy_summary

if embedding_results is None and Path("data/embeddings.pt").exists():
    embedding_results = torch.load("data/embeddings.pt", weights_only=False)
    print("✅ Loaded embeddings")

if embedding_results:
    print("🔄 Calculating entropy...")
    entropy_results = analyze_entropy(embedding_results)
    
    df = entropy_summary(entropy_results)
    print(f"\n✅ Analyzed {len(df)} sequences")
    print(f"📊 Global mean entropy: {entropy_results['global_mean']:.3f}")
    display(df)

In [ ]:
# Visualize entropy
import matplotlib.pyplot as plt

if entropy_results and len(entropy_results["entropy"]) > 0:
    vals = entropy_results["entropy"][0].float().numpy()
    
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(vals, alpha=0.7)
    ax.set_xlabel("Residue Position")
    ax.set_ylabel("Entropy (nats)")
    ax.set_title(f"Entropy: {entropy_results['sequence_id'][0]}")
    
    constrained = entropy_results["constrained_positions"][0].long().numpy()
    flexible = entropy_results["flexible_positions"][0].long().numpy()
    ax.scatter(constrained, vals[constrained], c="blue", s=10, alpha=0.5, label="Constrained")
    ax.scatter(flexible, vals[flexible], c="red", s=10, alpha=0.5, label="Flexible")
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## Step 4: Logits Analysis

In [ ]:
from src.analysis.logits_lib import analyze_residues, plot_heatmap, AA_VOCAB

residues_of_interest = {
    0: "Pos 1",
    10: "Pos 11",
    20: "Pos 21",
    50: "Pos 51"
}

if embedding_results:
    print("🔄 Analyzing logits...")
    logits_analysis = analyze_residues(
        embedding_results,
        residues_of_interest=residues_of_interest,
        pool_method="mean",
        scale_method="minmax"
    )
    print("✅ Complete")
    display(logits_analysis["probs"])

In [ ]:
if logits_analysis:
    plot_heatmap(
        logits_analysis["scaled_logits"],
        row_labels=logits_analysis["residue_labels"],
        col_labels=AA_VOCAB,
        title="Amino Acid Propensity",
        figsize=(12, 5)
    )

---
## Step 5: Export Results

In [ ]:
from google.colab import files
import shutil

output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

if 'entropy_results' in dir():
    entropy_summary(entropy_results).to_csv(output_dir / "entropy_summary.csv", index=False)
if 'logits_analysis' in dir():
    logits_analysis["probs"].to_csv(output_dir / "logits_probs.csv")
if embedding_results:
    save_embeddings(embedding_results, str(output_dir / "embeddings.pt"))

shutil.make_archive("results", "zip", "results")
files.download("results.zip")
print("📥 Downloading results.zip...")